# Multimodal Alzheimer's Disease Detection
## MRI (OASIS) + Audio (DementiaBank ADReSS) — Clinical Bridge Meta-Learner

**MRI preprocessing per group:**
- FreeSurfer (189-dim) → StandardScaler → PCA
- CNN embeddings (65536-dim) → StandardScaler → IncrementalPCA (memory-safe)
- Clinical (6-dim) → StandardScaler ONLY — no PCA

**Audio preprocessing per group:**
- openSMILE (6373-dim) → StandardScaler → PCA
- librosa (30-dim) → StandardScaler → PCA
- TF-IDF (500-dim) → StandardScaler → PCA
- Linguistic (11-dim) → StandardScaler ONLY — no PCA

**Meta-learner comparison:**
- Method 1: Static placeholder (0.5)
- Method 2: Dynamic placeholder (clinical logistic regression)
- Method 3: KNN soft imputation + entropy + dist (zero redundancy)

**Run cells top to bottom. CNN embeddings are saved after first run.**

## CELL 1 — Imports

In [1]:
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import nibabel as nib
import librosa
import opensmile
import joblib
from collections import defaultdict

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, IncrementalPCA
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_predict)
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve
from xgboost import XGBClassifier

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
SKF  = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print('Imports done.')

Imports done.


## CELL 2 — Paths

In [2]:
from config import (
    OASIS_ROOT,
    OASIS_CLINICAL,
    AUDIO_ROOT,
    TRANSCRIPT_ROOT,
    CC_META_PATH,
    CD_META_PATH,
    EMB_FILE,
    IDS_FILE,
)

print("Paths loaded successfully.")

Paths loaded successfully.


## CELL 3 — Shared helpers (report / XGBoost wrappers / OOF)

In [3]:
def report(name, y_true, y_prob):
    auc          = round(roc_auc_score(y_true, y_prob), 4)
    fpr, tpr, th = roc_curve(y_true, y_prob)
    t            = th[np.argmax(tpr - fpr)]
    acc          = round(accuracy_score(y_true, (y_prob >= t).astype(int)), 4)
    print(f'  {name:<52}  AUC={auc}  ACC={acc}')
    return auc, acc

def fit_xgb_mri(Xtr, Xte, ytr, yte, pw, name):
    m = XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.03,
                      subsample=0.8, colsample_bytree=0.8,
                      scale_pos_weight=pw, eval_metric='logloss',
                      random_state=SEED)
    m.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=False)
    p = m.predict_proba(Xte)[:, 1]
    auc, acc = report(name, yte, p)
    return m, p, auc, acc

def fit_xgb_audio(Xtr, Xte, ytr, yte, pw, name):
    m = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.8,
                      min_child_weight=5, reg_alpha=0.1, reg_lambda=1.5,
                      scale_pos_weight=pw, eval_metric='auc',
                      random_state=SEED)
    m.fit(Xtr, ytr, eval_set=[(Xte, yte)], verbose=False)
    p = m.predict_proba(Xte)[:, 1]
    auc, acc = report(name, yte, p)
    return m, p, auc, acc

def oof_probs(estimator, X, y, name):
    p = cross_val_predict(estimator, X, y,
                          cv=SKF, method='predict_proba', n_jobs=-1)[:, 1]
    auc, acc = report(name, y, p)
    return p, auc, acc

print('Helpers ready.')

Helpers ready.


---
# PART A — MRI PIPELINE (OASIS)

## CELL 4 — FreeSurfer parsing + dataframe

In [4]:
def parse_aseg(path):
    d = {}
    with open(path) as f:
        for line in f:
            if line.startswith('#'): continue
            p = line.split()
            if len(p) > 4: d[f'aseg_{p[4]}'] = float(p[3])
    return d

def parse_aparc(path, hemi):
    d = {}
    with open(path) as f:
        for line in f:
            if line.startswith('#'): continue
            p = line.split()
            if len(p) > 5:
                d[f'{hemi}_{p[0]}_area']  = float(p[2])
                d[f'{hemi}_{p[0]}_thick'] = float(p[4])
    return d

def build_fs_df(root):
    rows = []
    for disc in os.listdir(root):
        for subj in os.listdir(os.path.join(root, disc)):
            sp   = os.path.join(root, disc, subj, 'stats')
            aseg = os.path.join(sp, 'aseg.stats')
            lh   = os.path.join(sp, 'lh.aparc.stats')
            rh   = os.path.join(sp, 'rh.aparc.stats')
            if not os.path.exists(aseg): continue
            try:
                row = {'ID': subj}
                row.update(parse_aseg(aseg))
                row.update(parse_aparc(lh, 'lh'))
                row.update(parse_aparc(rh, 'rh'))
                rows.append(row)
            except: continue
    return pd.DataFrame(rows)

print('Parsing FreeSurfer stats...')
df_fs     = build_fs_df(OASIS_ROOT)
clin      = pd.read_excel(OASIS_CLINICAL)
clin['Gender'] = clin['M/F'].map({'M': 0, 'F': 1})
df        = df_fs.merge(clin, on='ID')
df['label'] = (df['CDR'] > 0).astype(int)

FS_COLS   = [c for c in df.columns if c.startswith(('aseg_','lh_','rh_'))]
CLIN_COLS = [c for c in ['Age','Gender','MMSE','eTIV','nWBV','ASF'] if c in df.columns]

X_fs_raw   = df[FS_COLS].values
X_clin_raw = df[CLIN_COLS].values
y_oasis    = df['label'].values

print(f'Subjects        : {len(df)}')
print(f'FreeSurfer dims : {X_fs_raw.shape[1]}')
print(f'Clinical dims   : {X_clin_raw.shape[1]}  (no PCA — small)')
print(f'Dem / Healthy   : {y_oasis.sum()} / {(y_oasis==0).sum()}')

Parsing FreeSurfer stats...
Subjects        : 425
FreeSurfer dims : 188
Clinical dims   : 6  (no PCA — small)
Dem / Healthy   : 93 / 332


## CELL 5 — MRI split + per-group preprocessing

In [5]:
mri_tr, mri_te = train_test_split(np.arange(len(df)), stratify=y_oasis,
                                   test_size=0.2, random_state=SEED)
y_mri_tr = y_oasis[mri_tr];  y_mri_te = y_oasis[mri_te]
pw_mri   = (y_mri_tr==0).sum() / max((y_mri_tr==1).sum(), 1)

# FreeSurfer: impute → scale → PCA  (fit on train only)
fs_imp = SimpleImputer(strategy='median')
fs_sc  = StandardScaler()
fs_pca = PCA(n_components=0.95, random_state=SEED)
X_fs_tr = fs_pca.fit_transform(fs_sc.fit_transform(fs_imp.fit_transform(X_fs_raw[mri_tr])))
X_fs_te = fs_pca.transform(fs_sc.transform(fs_imp.transform(X_fs_raw[mri_te])))
print(f'FreeSurfer after PCA : {X_fs_tr.shape[1]} components  (from {X_fs_raw.shape[1]})')

# Clinical: impute → scale ONLY  (no PCA — only 6 dims)
cl_imp = SimpleImputer(strategy='median')
cl_sc  = StandardScaler()
X_cl_tr = cl_sc.fit_transform(cl_imp.fit_transform(X_clin_raw[mri_tr]))
X_cl_te = cl_sc.transform(cl_imp.transform(X_clin_raw[mri_te]))
print(f'Clinical scaled      : {X_cl_tr.shape[1]} features  (no PCA)')

FreeSurfer after PCA : 94 components  (from 188)
Clinical scaled      : 6 features  (no PCA)


## CELL 6 — 3D CNN class + volume loader + trainer

In [6]:
class MRI3DCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv3d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(64,128,3,padding=1), nn.ReLU(), nn.MaxPool3d(2),
        )
        self.fc = nn.Sequential(
            nn.Linear(128*8*8*8, 256), nn.ReLU(), nn.Linear(256, 1))
    def forward(self, x):
        return self.fc(self.features(x).view(x.size(0), -1))

def load_mgz(path):
    vol = nib.load(path).get_fdata()
    vol = (vol - vol.mean()) / (vol.std() + 1e-6)
    t   = torch.tensor(vol, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    return F.interpolate(t, size=(64,64,64), mode='trilinear',
                         align_corners=False).squeeze(0)

def load_volumes(root, ld):
    vols, ids = [], []
    for disc in os.listdir(root):
        for subj in os.listdir(os.path.join(root, disc)):
            p = os.path.join(root, disc, subj, 'mri', 'brain.mgz')
            if not os.path.exists(p) or subj not in ld: continue
            try:
                vols.append(load_mgz(p)); ids.append(subj)
            except: continue
    X = torch.stack(vols)
    y = torch.tensor([ld[s] for s in ids], dtype=torch.float32).unsqueeze(1)
    return X, y, ids

def train_cnn(X, y, epochs=15, batch=8, lr=3e-4):
    model = MRI3DCNN()
    crit  = nn.BCEWithLogitsLoss()
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(len(X));  total = 0
        for i in range(0, len(X), batch):
            idx = perm[i:i+batch]
            opt.zero_grad()
            loss = crit(model(X[idx]), y[idx])
            loss.backward(); opt.step()
            total += loss.item()
        print(f'  CNN Epoch {ep+1:2d}/{epochs}  loss={total:.4f}')
    return model

print('CNN class ready.')

CNN class ready.


## CELL 7 — Train CNN or load saved embeddings

In [7]:
if os.path.exists(EMB_FILE):
    print('Loading saved CNN embeddings (skipping retraining)...')
    raw_emb = np.load(EMB_FILE)
    mri_ids = np.load(IDS_FILE, allow_pickle=True).tolist()
    print(f'  Loaded: {raw_emb.shape}')
else:
    print('Loading MRI volumes...')
    ld_dict               = dict(zip(df['ID'], df['label']))
    X_vol, y_vol, mri_ids = load_volumes(OASIS_ROOT, ld_dict)
    print(f'  {len(X_vol)} volumes. Training CNN...')
    cnn_model = train_cnn(X_vol, y_vol)
    print('  Extracting embeddings (batch=8)...')
    cnn_model.eval()
    parts = []
    with torch.no_grad():
        for i in range(0, len(X_vol), 8):
            xb   = X_vol[i:i+8]
            feat = cnn_model.features(xb)
            parts.append(feat.view(feat.size(0), -1).numpy())
            if i % 80 == 0: print(f'    {i}/{len(X_vol)}')
    raw_emb = np.vstack(parts)
    np.save(EMB_FILE, raw_emb)
    np.save(IDS_FILE, np.array(mri_ids))
    torch.save(cnn_model.state_dict(), 'cnn_weights.pth')
    print(f'  Saved: {raw_emb.shape}')

Loading saved CNN embeddings (skipping retraining)...
  Loaded: (425, 65536)


## CELL 8 — CNN preprocessing + MRI ablation study + MRI OOF

In [8]:
# Align embeddings with df (only subjects where both FS and CNN exist)
id2emb = {s: raw_emb[i] for i, s in enumerate(mri_ids)}
df_v   = df[df['ID'].isin(id2emb)].reset_index(drop=True)
Xfs_v  = df_v[FS_COLS].values
Xcl_v  = df_v[CLIN_COLS].values
y_v    = df_v['label'].values
Xemb_v = np.array([id2emb[s] for s in df_v['ID']])
print(f'Valid subjects (FS + CNN both present): {len(df_v)}')

trv, tev = train_test_split(np.arange(len(df_v)), stratify=y_v,
                             test_size=0.2, random_state=SEED)
y_trv = y_v[trv];  y_tev = y_v[tev]
pw_v  = (y_trv==0).sum() / max((y_trv==1).sum(), 1)

# Re-apply FS + Clinical pipelines (fitted in Cell 5)
Xfs_trv = fs_pca.transform(fs_sc.transform(fs_imp.transform(Xfs_v[trv])))
Xfs_tev = fs_pca.transform(fs_sc.transform(fs_imp.transform(Xfs_v[tev])))
Xcl_trv = cl_sc.transform(cl_imp.transform(Xcl_v[trv]))
Xcl_tev = cl_sc.transform(cl_imp.transform(Xcl_v[tev]))

# CNN: StandardScaler → IncrementalPCA  (memory-safe for 65536 dims)
cnn_sc   = StandardScaler()
cnn_ipca = IncrementalPCA(n_components=120, batch_size=200)
Xcnn_trv = cnn_ipca.fit_transform(cnn_sc.fit_transform(Xemb_v[trv]))
Xcnn_tev = cnn_ipca.transform(cnn_sc.transform(Xemb_v[tev]))
print(f'CNN after IncrementalPCA: {Xcnn_trv.shape[1]} components  (from 65536)')

print('\n' + '='*62)
print('MRI ABLATION STUDY  (held-out test set)')
print('='*62)
_, _, auc_a1, acc_a1 = fit_xgb_mri(
    np.hstack([Xfs_trv, Xcl_trv]), np.hstack([Xfs_tev, Xcl_tev]),
    y_trv, y_tev, pw_v, 'A1: FreeSurfer(PCA) + Clinical')
_, _, auc_a2, acc_a2 = fit_xgb_mri(
    np.hstack([Xcnn_trv, Xcl_trv]), np.hstack([Xcnn_tev, Xcl_tev]),
    y_trv, y_tev, pw_v, 'A2: CNN(IPCA) + Clinical')
_, _, auc_a3, acc_a3 = fit_xgb_mri(
    np.hstack([Xfs_trv, Xcnn_trv]), np.hstack([Xfs_tev, Xcnn_tev]),
    y_trv, y_tev, pw_v, 'A3: FreeSurfer(PCA) + CNN(IPCA)  [no Clinical]')
mri_model, _, auc_a4, acc_a4 = fit_xgb_mri(
    np.hstack([Xfs_trv, Xcnn_trv, Xcl_trv]),
    np.hstack([Xfs_tev, Xcnn_tev, Xcl_tev]),
    y_trv, y_tev, pw_v, 'A4: FreeSurfer(PCA)+CNN(IPCA)+Clinical  [FULL]')

print('\nMRI OOF cross-validation...')
Xfs_all  = fs_pca.transform(fs_sc.transform(fs_imp.transform(Xfs_v)))
Xcl_all  = cl_sc.transform(cl_imp.transform(Xcl_v))
Xcnn_all = cnn_ipca.transform(cnn_sc.transform(Xemb_v))
X_fusion = np.hstack([Xfs_all, Xcnn_all, Xcl_all])
y_fusion = y_v

mri_oof_prob, mri_oof_auc, mri_oof_acc = oof_probs(
    XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.03,
                  subsample=0.8, colsample_bytree=0.8,
                  scale_pos_weight=pw_v, eval_metric='logloss',
                  random_state=SEED),
    X_fusion, y_fusion, 'MRI Full Fusion OOF (unbiased)')

# Bridge: Age, Gender, MMSE only (first 3 clinical cols, already scaled)
mri_bridge = Xcl_all[:, :3]

Valid subjects (FS + CNN both present): 425
CNN after IncrementalPCA: 120 components  (from 65536)

MRI ABLATION STUDY  (held-out test set)
  A1: FreeSurfer(PCA) + Clinical                        AUC=0.9657  ACC=0.9059
  A2: CNN(IPCA) + Clinical                              AUC=0.9506  ACC=0.8706
  A3: FreeSurfer(PCA) + CNN(IPCA)  [no Clinical]        AUC=0.9226  ACC=0.7882
  A4: FreeSurfer(PCA)+CNN(IPCA)+Clinical  [FULL]        AUC=0.9625  ACC=0.9176

MRI OOF cross-validation...
  MRI Full Fusion OOF (unbiased)                        AUC=0.971  ACC=0.8894


---
# PART B — AUDIO PIPELINE (DementiaBank ADReSS)

## CELL 9 — Audio extraction functions

In [9]:
_smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.ComParE_2016,
    feature_level=opensmile.FeatureLevel.Functionals,
)

def extract_opensmile(path):
    try: return _smile.process_file(path).values.flatten()
    except: return None

def extract_librosa(path):
    try:
        y, sr    = librosa.load(path, sr=16000, mono=True)
        mfcc     = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13), axis=1)
        chroma   = np.mean(librosa.feature.chroma_stft(y=y, sr=sr),     axis=1)
        centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        zcr      = np.mean(librosa.feature.zero_crossing_rate(y))
        rms      = np.mean(librosa.feature.rms(y=y))
        rolloff  = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
        bw       = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr))
        return np.hstack([mfcc, chroma, centroid, zcr, rms, rolloff, bw])
    except: return None

def parse_cha(path):
    lines, all_words = [], []
    wc = sc = fc = 0
    fillers  = {'&uh','&um','&er','&ah'}
    pronouns = {'i','he','she','they','we','it','you','me','him','her'}
    cookie   = ['woman','washing','dishes','kitchen','boy','stool',
                'cookie','jar','falling','water','overflowing',
                'sink','window','curtains','outside']
    with open(path, encoding='utf8', errors='ignore') as f:
        for line in f:
            if line.startswith('*PAR:') or line.startswith('*INV:'):
                line = line.split(':',1)[-1].strip()
                lines.append(line)
                ws = line.split()
                all_words.extend(ws)
                wc += len(ws);  sc += 1
                fc += sum(1 for w in ws if w in fillers)
    text  = ' '.join(lines)
    tlow  = text.lower()
    vocab = set(all_words)
    info  = sum(1 for u in cookie if u in tlow)
    ling  = np.array([wc, sc, fc,
                      len(vocab)/(wc+1e-6), wc/(sc+1e-6), fc/(wc+1e-6),
                      sum(1 for w in all_words if w.lower() in pronouns),
                      info,
                      sum(1 for w in vocab if all_words.count(w)==1)/(len(vocab)+1e-6),
                      len(vocab), wc/(info+1e-6)], dtype=float)
    return text, ling

def load_adress_meta(cc_path, cd_path):
    def _p(path, lbl):
        rows = []
        with open(path) as f:
            for line in f.readlines()[1:]:
                p = line.strip().split(';')
                if len(p) < 4: continue
                sid, age, gender, mmse = [x.strip() for x in p[:4]]
                rows.append({'ID': sid,
                             'Age': float(age) if age.replace('.','').isdigit() else np.nan,
                             'Gender': 0 if gender=='male' else 1,
                             'MMSE':  float(mmse) if mmse!='NA' else np.nan,
                             'label': lbl})
        return pd.DataFrame(rows)
    meta = pd.concat([_p(cc_path,0), _p(cd_path,1)], ignore_index=True)
    meta['MMSE'] = meta['MMSE'].fillna(meta['MMSE'].median())
    return meta

print('Audio functions ready.')

Audio functions ready.


## CELL 10 — Extract all audio features

In [10]:
meta_df = load_adress_meta(CC_META_PATH, CD_META_PATH)
print(f'ADReSS meta: CC={( meta_df.label==0).sum()}  CD={(meta_df.label==1).sum()}')
print('Extracting features (openSMILE ~2-5s per file)...')

subj_smile   = defaultdict(list)
subj_librosa = defaultdict(list)
subj_texts, subj_ling, subj_bridge, subj_labels = {}, {}, {}, {}
skipped = 0

for folder in ['cc', 'cd']:
    af = os.path.join(AUDIO_ROOT, folder)
    tf = os.path.join(TRANSCRIPT_ROOT, folder)
    for fname in sorted(os.listdir(af)):
        if not fname.endswith('.wav'): continue
        sid = fname.split('.')[0]
        ap  = os.path.join(af, fname)
        tp  = os.path.join(tf, sid + '.cha')
        if not os.path.exists(tp): skipped += 1; continue
        row = meta_df[meta_df['ID'] == sid]
        if row.empty: skipped += 1; continue
        fs  = extract_opensmile(ap)
        fl  = extract_librosa(ap)
        if fs is None or fl is None: skipped += 1; continue
        subj_smile[sid].append(fs)
        subj_librosa[sid].append(fl)
        if sid not in subj_texts:
            text, ling         = parse_cha(tp)
            subj_texts[sid]    = text
            subj_ling[sid]     = ling
            subj_bridge[sid]   = row[['Age','Gender','MMSE']].values.flatten().astype(float)
            subj_labels[sid]   = int(row['label'].values[0])

records = []
for sid in subj_smile:
    records.append({'sid': sid,
                    'smile':   np.array(subj_smile[sid]).mean(axis=0),
                    'librosa': np.array(subj_librosa[sid]).mean(axis=0),
                    'text':    subj_texts[sid],
                    'ling':    subj_ling[sid],
                    'bridge':  subj_bridge[sid],
                    'label':   subj_labels[sid]})

df_audio     = pd.DataFrame(records)
audio_labels = df_audio['label'].values
print(f'Extracted: {len(df_audio)} subjects  skipped={skipped}')
print(f'Label dist: {dict(pd.Series(audio_labels).value_counts().sort_index())}')

ADReSS meta: CC=54  CD=54
Extracting features (openSMILE ~2-5s per file)...
Extracted: 108 subjects  skipped=0
Label dist: {0: 54, 1: 54}


## CELL 11 — Audio split + per-group preprocessing + ablation

In [11]:
# Split FIRST
atr_idx, ate_idx = train_test_split(df_audio.index, stratify=audio_labels,
                                     test_size=0.2, random_state=SEED)
tr_df = df_audio.loc[atr_idx].reset_index(drop=True)
te_df = df_audio.loc[ate_idx].reset_index(drop=True)
ya_tr = tr_df['label'].values;  ya_te = te_df['label'].values
pw_a  = (ya_tr==0).sum() / max((ya_tr==1).sum(), 1)
print(f'Train: {len(tr_df)}  Test: {len(te_df)}')

sm_tr = np.vstack(tr_df['smile'].values);    sm_te = np.vstack(te_df['smile'].values)
lb_tr = np.vstack(tr_df['librosa'].values);  lb_te = np.vstack(te_df['librosa'].values)
lg_tr = np.vstack(tr_df['ling'].values);     lg_te = np.vstack(te_df['ling'].values)
bridge_tr = np.vstack(tr_df['bridge'].values)   # (N_tr, 3)
bridge_te = np.vstack(te_df['bridge'].values)
# TF-IDF fit on TRAIN only
vec   = TfidfVectorizer(max_features=500, ngram_range=(1,2),
                        sublinear_tf=True, token_pattern=r'(?u)\b\w+\b')
tf_tr = vec.fit_transform(tr_df['text'].tolist()).toarray()
tf_te = vec.transform(te_df['text'].tolist()).toarray()

# openSMILE: scale → PCA
sm_sc=StandardScaler(); sm_pca=PCA(n_components=0.95, random_state=SEED)
sm_tr_p=sm_pca.fit_transform(sm_sc.fit_transform(sm_tr))
sm_te_p=sm_pca.transform(sm_sc.transform(sm_te))
print(f'openSMILE after PCA : {sm_tr_p.shape[1]} components  (from {sm_tr.shape[1]})')

# librosa: scale → PCA
lb_sc=StandardScaler(); lb_pca=PCA(n_components=0.95, random_state=SEED)
lb_tr_p=lb_pca.fit_transform(lb_sc.fit_transform(lb_tr))
lb_te_p=lb_pca.transform(lb_sc.transform(lb_te))
print(f'librosa after PCA   : {lb_tr_p.shape[1]} components  (from {lb_tr.shape[1]})')

# TF-IDF: scale → PCA
tf_sc=StandardScaler(); tf_pca=PCA(n_components=0.95, random_state=SEED)
tf_tr_p=tf_pca.fit_transform(tf_sc.fit_transform(tf_tr))
tf_te_p=tf_pca.transform(tf_sc.transform(tf_te))
print(f'TF-IDF after PCA    : {tf_tr_p.shape[1]} components  (from {tf_tr.shape[1]})')

# Linguistic: scale ONLY (11 dims)
lg_sc=StandardScaler()
lg_tr_s=lg_sc.fit_transform(lg_tr)
lg_te_s=lg_sc.transform(lg_te)
print(f'Linguistic scaled   : {lg_tr_s.shape[1]} features  (no PCA)')

br_sc    = StandardScaler()
bridge_tr_s = br_sc.fit_transform(bridge_tr)    # scale only — no PCA (3 dims)
bridge_te_s = br_sc.transform(bridge_te)

print('\n' + '='*62)
print('AUDIO ABLATION STUDY  (held-out test set)')
print('='*62)
_, _, auc_b1, acc_b1 = fit_xgb_audio(
    sm_tr_p, sm_te_p, ya_tr, ya_te, pw_a, 'B1: openSMILE(PCA) only')
_, _, auc_b2, acc_b2 = fit_xgb_audio(
    lb_tr_p, lb_te_p, ya_tr, ya_te, pw_a, 'B2: librosa(PCA) only')
_, _, auc_b3, acc_b3 = fit_xgb_audio(
    np.hstack([tf_tr_p,lg_tr_s]), np.hstack([tf_te_p,lg_te_s]),
    ya_tr, ya_te, pw_a, 'B3: TF-IDF(PCA) + Linguistic  [text only]')
_, _, auc_b4, acc_b4 = fit_xgb_audio(
    np.hstack([sm_tr_p,lb_tr_p]), np.hstack([sm_te_p,lb_te_p]),
    ya_tr, ya_te, pw_a, 'B4: openSMILE(PCA)+librosa(PCA)  [acoustic only]')
audio_model, _, auc_b5, acc_b5 = fit_xgb_audio(
    np.hstack([sm_tr_p,lb_tr_p,tf_tr_p,lg_tr_s]),
    np.hstack([sm_te_p,lb_te_p,tf_te_p,lg_te_s]),
    ya_tr, ya_te, pw_a, 'B5: openSMILE+librosa+TF-IDF+clinincal')


Train: 86  Test: 22
openSMILE after PCA : 65 components  (from 6373)
librosa after PCA   : 16 components  (from 30)
TF-IDF after PCA    : 72 components  (from 500)
Linguistic scaled   : 11 features  (no PCA)

AUDIO ABLATION STUDY  (held-out test set)
  B1: openSMILE(PCA) only                               AUC=0.4132  ACC=0.5455
  B2: librosa(PCA) only                                 AUC=0.6529  ACC=0.6818
  B3: TF-IDF(PCA) + Linguistic  [text only]             AUC=0.8182  ACC=0.8182
  B4: openSMILE(PCA)+librosa(PCA)  [acoustic only]      AUC=0.4132  ACC=0.5455
  B5: openSMILE+librosa+TF-IDF+clinincal                AUC=0.7603  ACC=0.7727


## CELL 12 — Audio OOF (full no-leakage fold preprocessing)

In [12]:
print('Generating Audio OOF (preprocessing re-fit inside each fold)...')
sm_all = np.vstack(df_audio['smile'].values)
lb_all = np.vstack(df_audio['librosa'].values)
lg_all = np.vstack(df_audio['ling'].values)
tx_all = df_audio['text'].tolist()
br_all = np.vstack(df_audio['bridge'].values)

audio_oof_prob = np.zeros(len(df_audio))

for fold, (tri, vai) in enumerate(SKF.split(sm_all, audio_labels)):
    s1=StandardScaler(); p1=PCA(n_components=0.95,random_state=SEED)
    st=p1.fit_transform(s1.fit_transform(sm_all[tri]))
    sv=p1.transform(s1.transform(sm_all[vai]))
    s2=StandardScaler(); p2=PCA(n_components=0.95,random_state=SEED)
    lt=p2.fit_transform(s2.fit_transform(lb_all[tri]))
    lv=p2.transform(s2.transform(lb_all[vai]))
    vf=TfidfVectorizer(max_features=500,ngram_range=(1,2),
                       sublinear_tf=True,token_pattern=r'(?u)\b\w+\b')
    tfr=vf.fit_transform([tx_all[i] for i in tri]).toarray()
    tfv=vf.transform([tx_all[i] for i in vai]).toarray()
    s3=StandardScaler(); p3=PCA(n_components=0.95,random_state=SEED)
    tt=p3.fit_transform(s3.fit_transform(tfr))
    tv=p3.transform(s3.transform(tfv))
    s4=StandardScaler()
    gt=s4.fit_transform(lg_all[tri])
    gv=s4.transform(lg_all[vai])
    Xf_tr=np.hstack([st,lt,tt,gt]);  Xf_va=np.hstack([sv,lv,tv,gv])
    yf_tr=audio_labels[tri]
    pw=(yf_tr==0).sum()/max((yf_tr==1).sum(),1)
    
    fm=XGBClassifier(n_estimators=300,max_depth=3,learning_rate=0.05,
                     subsample=0.8,colsample_bytree=0.8,min_child_weight=5,
                     scale_pos_weight=pw,reg_alpha=0.1,reg_lambda=1.5,
                     eval_metric='auc',random_state=SEED)
    fm.fit(Xf_tr, yf_tr, verbose=False)
    audio_oof_prob[vai] = fm.predict_proba(Xf_va)[:,1]
    fa = round(roc_auc_score(audio_labels[vai], audio_oof_prob[vai]), 4)
    print(f'  Fold {fold+1}: AUC={fa}  N_val={len(vai)}')

oof_audio_auc, oof_audio_acc = report(
    'Audio Full Fusion OOF (unbiased)', audio_labels, audio_oof_prob)
audio_bridge = br_all   # (108, 3) raw Age, Gender, MMSE

Generating Audio OOF (preprocessing re-fit inside each fold)...
  Fold 1: AUC=0.8926  N_val=22
  Fold 2: AUC=0.9669  N_val=22
  Fold 3: AUC=0.9174  N_val=22
  Fold 4: AUC=0.8636  N_val=21
  Fold 5: AUC=0.8818  N_val=21
  Audio Full Fusion OOF (unbiased)                      AUC=0.892  ACC=0.8426


---
# PART C — META-LEARNER

## CELL 13 — KNN imputation function

In [13]:
K = 7  

def knn_impute(query_bridge, library_bridge,
               library_probs, library_labels, k=K):
    sc  = StandardScaler()
    lib = sc.fit_transform(library_bridge[:, [0, 2]])   # Age col, MMSE col
    qry = sc.transform(query_bridge[:,    [0, 2]])

    nn = NearestNeighbors(n_neighbors=k, metric='euclidean')
    nn.fit(lib)
    dists, idxs = nn.kneighbors(qry)

    imputed   = np.zeros(len(qry))
    entropy   = np.zeros(len(qry))
    mean_dist = np.zeros(len(qry))

    for i in range(len(qry)):
        idx = idxs[i];  d = dists[i]
        w   = 1.0 / (d + 1e-6);  w /= w.sum()
        imputed[i]   = np.dot(w, library_probs[idx])
        p_dem        = library_labels[idx].mean()
        entropy[i]   = (-p_dem*np.log(p_dem+1e-9)
                        -(1-p_dem)*np.log(1-p_dem+1e-9))
        mean_dist[i] = d.mean()

    return imputed, entropy, mean_dist

print(f'KNN function ready  (K={K})')

KNN function ready  (K=7)


## CELL 14 — Build all three meta-datasets

In [27]:
N_mri   = len(mri_oof_prob)
N_audio = len(audio_oof_prob)
y_meta  = np.concatenate([y_fusion, audio_labels])

# Consistent bridge scaling
br_oasis  = mri_bridge                           # (N_mri, 3) — already scaled
br_adress = StandardScaler().fit_transform(
                SimpleImputer(strategy='median')
                .fit_transform(audio_bridge))    # (108, 3)

print(f'OASIS subjects  : {N_mri}')
print(f'ADReSS subjects : {N_audio}')
print(f'Total meta rows : {N_mri + N_audio}')

# ── METHOD 1: Static 0.5 ─────────────────────────────────────────────────────
X_static = np.vstack([
    np.column_stack([mri_oof_prob,           np.full(N_mri,  0.5)]),
    np.column_stack([np.full(N_audio, 0.5),  audio_oof_prob]),
])
sc1=StandardScaler(); X_static_sc=sc1.fit_transform(X_static)

# ── METHOD 2: Dynamic — clinical logistic regression placeholder ──────────────
X_clin_comb = np.vstack([br_oasis, br_adress])
clin_oof = cross_val_predict(
    LogisticRegression(max_iter=2000, class_weight='balanced',
                       C=1.0, random_state=SEED),
    StandardScaler().fit_transform(X_clin_comb),
    y_meta, cv=SKF, method='predict_proba', n_jobs=-1)[:, 1]
P_clin_oasis  = clin_oof[:N_mri]
P_clin_adress = clin_oof[N_mri:]

X_dynamic = np.vstack([
    np.column_stack([mri_oof_prob,   P_clin_oasis]),
    np.column_stack([P_clin_adress,  audio_oof_prob]),
])
sc2=StandardScaler(); X_dynamic_sc=sc2.fit_transform(X_dynamic)

# ── METHOD 3: KNN — 6 features, ZERO redundancy ──────────────────────────────
# Age/Gender/MMSE EXCLUDED from bridge vector because:
#   P_mri   already encodes them (trained on FS + CNN + Age + Gender + MMSE)
#   P_audio already encodes them (trained on openSMILE + librosa + TF-IDF + MMSE)
#   Adding again = multicollinearity
# Bridge = [P_real, P_knn, has_mri, has_audio, entropy, dist]
print(f'\nKNN imputation (K={K})...')

P_audio_knn, ent_oasis,  dist_oasis  = knn_impute(
    br_oasis, br_adress, audio_oof_prob, audio_labels)
P_mri_knn,   ent_adress, dist_adress = knn_impute(
    br_adress, br_oasis, mri_oof_prob, y_fusion)

print(f'  P_audio_knn range : [{P_audio_knn.min():.3f}, {P_audio_knn.max():.3f}]')
print(f'  P_mri_knn   range : [{P_mri_knn.min():.3f},  {P_mri_knn.max():.3f}]')
print(f'  Entropy OASIS     : [{ent_oasis.min():.3f}, {ent_oasis.max():.3f}]  (0=reliable, 0.693=50/50)')

X_knn = np.vstack([
    np.column_stack([mri_oof_prob,   P_audio_knn,
                     np.ones(N_mri), np.zeros(N_mri),
                     ent_oasis,      dist_oasis]),
    np.column_stack([P_mri_knn,      audio_oof_prob,
                     np.zeros(N_audio), np.ones(N_audio),
                     ent_adress,     dist_adress]),  
])
sc3=StandardScaler(); X_knn_sc=sc3.fit_transform(X_knn)

print(f'\nBridge feature counts:')
print(f'  Static  : {X_static.shape[1]}  [P_mri, P_audio]')
print(f'  Dynamic : {X_dynamic.shape[1]}  [P_mri, P_clin  OR  P_clin, P_audio]')
print(f'  KNN     : {X_knn.shape[1]}  [P_real, P_knn, has_mri, has_audio, entropy, dist]')

OASIS subjects  : 425
ADReSS subjects : 108
Total meta rows : 533

KNN imputation (K=7)...
  P_audio_knn range : [0.169, 0.927]
  P_mri_knn   range : [0.002,  0.983]
  Entropy OASIS     : [-0.000, 0.683]  (0=reliable, 0.693=50/50)

Bridge feature counts:
  Static  : 2  [P_mri, P_audio]
  Dynamic : 2  [P_mri, P_clin  OR  P_clin, P_audio]
  KNN     : 6  [P_real, P_knn, has_mri, has_audio, entropy, dist]


## CELL 15 — Evaluate all three meta-learners via OOF

In [28]:
base_lr = LogisticRegression(max_iter=2000, class_weight='balanced',
                              C=0.5, random_state=SEED)

def meta_oof_eval(X_sc, y, name):
    p = cross_val_predict(base_lr, X_sc, y,
                          cv=SKF, method='predict_proba', n_jobs=-1)[:, 1]
    auc, acc = report(name, y, p)
    return p, auc, acc

print('='*62)
print('META-LEARNER OOF COMPARISON')
print('='*62)
oof_s, auc_s, acc_s = meta_oof_eval(X_static_sc,  y_meta, 'Meta Static (0.5)')
oof_d, auc_d, acc_d = meta_oof_eval(X_dynamic_sc, y_meta, 'Meta Dynamic (clinical model)')
oof_k, auc_k, acc_k = meta_oof_eval(X_knn_sc,     y_meta,
                                     'Meta KNN + entropy + dist  [no redundancy]')

# Auto-select best by OOF AUC
all_aucs  = [auc_s,       auc_d,        auc_k]
all_names = ['Static',    'Dynamic',    'KNN']
all_Xs    = [X_static_sc, X_dynamic_sc, X_knn_sc]
all_scs   = [sc1,         sc2,          sc3]
all_oofs  = [oof_s,       oof_d,        oof_k]
best_i    = int(np.argmax(all_aucs))
print(f'\n  Best method: {all_names[best_i]}  (AUC={all_aucs[best_i]})')

final_meta = CalibratedClassifierCV(base_lr, cv=5, method='isotonic')
final_meta.fit(all_Xs[best_i], y_meta)
meta_oof_prob = all_oofs[best_i]

META-LEARNER OOF COMPARISON
  Meta Static (0.5)                                     AUC=0.9517  ACC=0.8931
  Meta Dynamic (clinical model)                         AUC=0.9599  ACC=0.9081
  Meta KNN + entropy + dist  [no redundancy]            AUC=0.9425  ACC=0.9024

  Best method: Dynamic  (AUC=0.9599)


## CELL 17 — Save all models

In [29]:
joblib.dump({'fs_imp':fs_imp,'fs_sc':fs_sc,'fs_pca':fs_pca,
             'cl_imp':cl_imp,'cl_sc':cl_sc,
             'cnn_sc':cnn_sc,'cnn_ipca':cnn_ipca,
             'mri_model':mri_model,
             'FS_COLS':FS_COLS,'CLIN_COLS':CLIN_COLS},
            'mri_pipeline.pkl')

joblib.dump({'vec':vec,
             'sm_sc':sm_sc,'sm_pca':sm_pca,
             'lb_sc':lb_sc,'lb_pca':lb_pca,
             'tf_sc':tf_sc,'tf_pca':tf_pca,
             'lg_sc':lg_sc,
             'audio_model':audio_model},
            'audio_pipeline.pkl')

joblib.dump({'meta_scaler':all_scs[best_i],
             'final_meta':final_meta,
             'best_method':all_names[best_i],
             'K':K,
             'br_oasis':br_oasis,'br_adress':br_adress,
             'mri_oof_prob':mri_oof_prob,
             'audio_oof_prob':audio_oof_prob,
             'y_fusion':y_fusion,'audio_labels':audio_labels},
            'meta_pipeline.pkl')

print('Saved: mri_pipeline.pkl  audio_pipeline.pkl  meta_pipeline.pkl')



Saved: mri_pipeline.pkl  audio_pipeline.pkl  meta_pipeline.pkl


In [130]:
#Demo 

In [30]:
DEMO_MRI_SUBJECT   = "OAS1_0425_MR1"  
DEMO_AUDIO_SUBJECT = "S007"            


OASIS_ROOT      = r"D:\ML Project\ALZHEIMERS\OASIS\OASIS_dis"
OASIS_CLINICAL  = r"D:\ML Project\ALZHEIMERS\OASIS\oasis_cross-sectional-5708aa0a98d82080.xlsx"
AUDIO_ROOT      = r"D:\ML Project\Dimentia Bank\ADReSS-IS2020-train\ADReSS-IS2020-data\train\Full_wave_enhanced_audio"
TRANSCRIPT_ROOT = r"D:\ML Project\Dimentia Bank\ADReSS-IS2020-train\ADReSS-IS2020-data\train\transcription"
CC_META_PATH    = r"D:\ML Project\Dimentia Bank\ADReSS-IS2020-train\ADReSS-IS2020-data\train\cc_meta_data.txt"
CD_META_PATH    = r"D:\ML Project\Dimentia Bank\ADReSS-IS2020-train\ADReSS-IS2020-data\train\cd_meta_data.txt"


import joblib
import numpy as np
from IPython.display import display, HTML

_meta          = joblib.load("meta_pipeline.pkl")
meta_scaler    = _meta["meta_scaler"]
final_meta     = _meta["final_meta"]
K              = _meta["K"]
br_oasis       = _meta["br_oasis"]
br_adress      = _meta["br_adress"]
mri_oof_prob   = _meta["mri_oof_prob"]
audio_oof_prob = _meta["audio_oof_prob"]
audio_labels   = _meta["audio_labels"]

def show_box(title, lines, border="#1A7B8A", bg="#0F172A"):
    rows = "".join(
        f"<tr><td style='color:#94A3B8;padding:4px 14px;white-space:nowrap'>{k}</td>"
        f"<td style='color:#F1F5F9;padding:4px 14px;font-family:Consolas'>{v}</td></tr>"
        for k, v in lines
    )
    display(HTML(
        f"<div style='background:{bg};border-left:4px solid {border};"
        f"border-radius:6px;margin:8px 0;font-family:Segoe UI,sans-serif'>"
        f"<div style='background:{border}22;padding:8px 14px;font-weight:700;"
        f"color:{border};font-size:1em'>{title}</div>"
        f"<table style='border-collapse:collapse;width:100%'>{rows}</table></div>"
    ))

def show_result(label, prob, true_label):
    verdict = "DEMENTIA" if prob > 0.5 else "HEALTHY"
    correct = (prob > 0.5) == bool(true_label)
    pcol    = "#EF4444" if prob > 0.5 else "#10B981"
    tick    = "correct" if correct else "misclassified"
    tcol    = "#10B981" if correct else "#EF4444"
    display(HTML(
        f"<div style='background:#1E293B;border-radius:8px;padding:18px 22px;"
        f"margin:10px 0;font-family:Segoe UI,sans-serif;display:flex;"
        f"align-items:center;gap:32px'>"
        f"<div><div style='color:#94A3B8;font-size:0.82em'>{label}</div>"
        f"<div style='color:{pcol};font-size:2.8em;font-weight:700'>{prob:.4f}</div></div>"
        f"<div><div style='color:{pcol};font-size:1.2em;font-weight:700'>{verdict}</div>"
        f"<div style='color:{tcol};font-size:0.9em;margin-top:4px'>"
        f"Ground truth: {'DEMENTIA' if true_label else 'HEALTHY'} -- {tick}</div></div>"
        f"</div>"
    ))

print(f"MRI subject  : {DEMO_MRI_SUBJECT}")
print(f"Audio subject: {DEMO_AUDIO_SUBJECT}")
print(f"K            : {K}")
print(f"Meta method  : {_meta['best_method']}")
print("Ready.")

MRI subject  : OAS1_0425_MR1
Audio subject: S007
K            : 7
Meta method  : Dynamic
Ready.


In [31]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import nibabel as nib

print("=" * 60)
print("STAGE A — MRI PIPELINE")
print("=" * 60)

# ── Locate subject folder ─────────────────────────────────────────────────────
subj_dir = None
for disc in os.listdir(OASIS_ROOT):
    candidate = os.path.join(OASIS_ROOT, disc, DEMO_MRI_SUBJECT)
    if os.path.isdir(candidate):
        subj_dir = candidate
        break
assert subj_dir, f"Subject '{DEMO_MRI_SUBJECT}' not found in {OASIS_ROOT}"
print(f"✓ Located  : {subj_dir}")

# ── FreeSurfer stats ──────────────────────────────────────────────────────────
sp  = os.path.join(subj_dir, "stats")
row = {"ID": DEMO_MRI_SUBJECT}
row.update(parse_aseg(os.path.join(sp, "aseg.stats")))
row.update(parse_aparc(os.path.join(sp, "lh.aparc.stats"), "lh"))
row.update(parse_aparc(os.path.join(sp, "rh.aparc.stats"), "rh"))
print(f"✓ FreeSurfer: {len(row)-1} morphometric features parsed")

# ── Clinical variables ────────────────────────────────────────────────────────
clin_df      = pd.read_excel(OASIS_CLINICAL)
clin_df["Gender"] = clin_df["M/F"].map({"M": 0, "F": 1})
clin_row     = clin_df[clin_df["ID"] == DEMO_MRI_SUBJECT].iloc[0]
demo_true_mri = int(clin_row["CDR"] > 0)
demo_age_mri  = float(clin_row["Age"])
demo_mmse_mri = float(clin_row["MMSE"])

show_box("Subject Clinical Info", [
    ("Subject ID",  DEMO_MRI_SUBJECT),
    ("Age",         f"{demo_age_mri:.0f} years"),
    ("MMSE",        f"{demo_mmse_mri:.0f}  (30 = perfect, <24 = possible impairment)"),
    ("CDR",         f"{clin_row['CDR']}"),
    ("True label",  "DEMENTIA" if demo_true_mri else "HEALTHY"),
], border="#2E75B6")


if DEMO_MRI_SUBJECT in id2emb:
    raw_emb = id2emb[DEMO_MRI_SUBJECT].reshape(1, -1)
else:
    print("\n  Subject not in id2emb — running CNN inference on brain.mgz ...")
    _cnn = MRI3DCNN()
    _weights_file = "cnn_weights.pth"
    assert os.path.exists(_weights_file), (
        f"\n  ERROR: {_weights_file} not found.\n"
    )
    _cnn.load_state_dict(torch.load(_weights_file, map_location="cpu"))
    _cnn.eval()
    mgz_path = os.path.join(subj_dir, "mri", "brain.mgz")
    vol      = nib.load(mgz_path).get_fdata()
    vol      = (vol - vol.mean()) / (vol.std() + 1e-6)
    t        = torch.tensor(vol, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    vol_t    = F.interpolate(t, size=(64,64,64), mode="trilinear",
                             align_corners=False).squeeze(0)
    with torch.no_grad():
        raw_emb = _cnn.features(vol_t.unsqueeze(0)).view(1, -1).numpy()
    print(f"  ✓ CNN inference done")

print(f"✓ CNN embedding shape: {raw_emb.shape}  (65,536-dim raw features)")

# ── Align to training columns & preprocess ────────────────────────────────────
df_row   = pd.DataFrame([row]).reindex(columns=FS_COLS, fill_value=np.nan)
X_fs_raw = df_row.values
X_cl_raw = np.array([[clin_row.get(c, np.nan) for c in CLIN_COLS]])

X_fs  = fs_pca.transform(fs_sc.transform(fs_imp.transform(X_fs_raw)))
X_cl  = cl_sc.transform(cl_imp.transform(X_cl_raw))
X_cnn = cnn_ipca.transform(cnn_sc.transform(raw_emb))
X_mri = np.hstack([X_fs, X_cnn, X_cl])

print(f"\n✓ Preprocessed and fused:")
print(f"  FreeSurfer (PCA)       : {X_fs.shape[1]} components")
print(f"  CNN (IncrementalPCA)   : {X_cnn.shape[1]} components  (from 65,536)")
print(f"  Clinical               : {X_cl.shape[1]} features  (no PCA — kept interpretable)")
print(f"  XGBoost input          : {X_mri.shape[1]}-dim fused vector")

# ── XGBoost prediction ────────────────────────────────────────────────────────
print("\nRunning XGBoost MRI fusion classifier ...")
p_mri_demo = float(mri_model.predict_proba(X_mri)[0, 1])

print()
show_result("P(dementia | MRI)", p_mri_demo, demo_true_mri)

# Save Age/MMSE for Stage C bridge
demo_bridge_mri = np.array([[demo_age_mri, 0.0, demo_mmse_mri]])


STAGE A — MRI PIPELINE
✓ Located  : D:\ML Project\ALZHEIMERS\OASIS\OASIS_dis\disc11\OAS1_0425_MR1
✓ FreeSurfer: 188 morphometric features parsed


Subject ID,OAS1_0425_MR1
Age,78 years
MMSE,"23 (30 = perfect, <24 = possible impairment)"
CDR,1.0
True label,DEMENTIA


✓ CNN embedding shape: (1, 65536)  (65,536-dim raw features)

✓ Preprocessed and fused:
  FreeSurfer (PCA)       : 94 components
  CNN (IncrementalPCA)   : 120 components  (from 65,536)
  Clinical               : 6 features  (no PCA — kept interpretable)
  XGBoost input          : 220-dim fused vector

Running XGBoost MRI fusion classifier ...



In [32]:
print("=" * 60)
print("STAGE B — AUDIO PIPELINE")
print("=" * 60)

# ── Locate WAV + transcript ───────────────────────────────────────────────────
audio_folder = None
for folder in ["cc", "cd"]:
    wav_path = os.path.join(AUDIO_ROOT, folder, DEMO_AUDIO_SUBJECT + ".wav")
    if os.path.exists(wav_path):
        audio_folder = folder
        break
assert audio_folder, f"Subject '{DEMO_AUDIO_SUBJECT}' not found in cc/ or cd/"

audio_path  = os.path.join(AUDIO_ROOT,      audio_folder, DEMO_AUDIO_SUBJECT + ".wav")
transc_path = os.path.join(TRANSCRIPT_ROOT, audio_folder, DEMO_AUDIO_SUBJECT + ".cha")
demo_true_audio = 1 if audio_folder == "cd" else 0
print(f"✓ Located  : {audio_folder}/{DEMO_AUDIO_SUBJECT}.wav")

# ── Clinical from meta file ───────────────────────────────────────────────────
meta_all = load_adress_meta(CC_META_PATH, CD_META_PATH)
meta_row = meta_all[meta_all["ID"] == DEMO_AUDIO_SUBJECT]
demo_age_audio  = float(meta_row.iloc[0]["Age"])  if not meta_row.empty else 70.0
demo_mmse_audio = float(meta_row.iloc[0]["MMSE"]) if not meta_row.empty else 20.0

show_box("Subject Clinical Info", [
    ("Subject ID",  DEMO_AUDIO_SUBJECT),
    ("Folder",      f"{audio_folder}/  (cc=Healthy · cd=Dementia)"),
    ("Age",         f"{demo_age_audio:.0f} years"),
    ("MMSE",        f"{demo_mmse_audio:.0f}"),
    ("True label",  "DEMENTIA" if demo_true_audio else "HEALTHY"),
], border="#C4501A")

# ── Feature extraction ────────────────────────────────────────────────────────
print("\nExtracting openSMILE ComParE (6,373 acoustic functionals) ...", end=" ")
sm = _smile.process_file(audio_path).values.flatten()
print(f"done  shape: {sm.shape}")

print("Extracting librosa spectral features ...", end=" ")
import librosa
y_wav, sr    = librosa.load(audio_path, sr=16000, mono=True)
mfcc     = np.mean(librosa.feature.mfcc(y=y_wav, sr=sr, n_mfcc=13), axis=1)
chroma   = np.mean(librosa.feature.chroma_stft(y=y_wav, sr=sr), axis=1)
centroid = np.mean(librosa.feature.spectral_centroid(y=y_wav, sr=sr))
zcr      = np.mean(librosa.feature.zero_crossing_rate(y_wav))
rms      = np.mean(librosa.feature.rms(y=y_wav))
rolloff  = np.mean(librosa.feature.spectral_rolloff(y=y_wav, sr=sr))
bw       = np.mean(librosa.feature.spectral_bandwidth(y=y_wav, sr=sr))
lb = np.hstack([mfcc, chroma, centroid, zcr, rms, rolloff, bw])
print(f"done  shape: {lb.shape}")

print("Parsing CHAT transcript + handcrafted linguistic features ...", end=" ")
text, ling = parse_cha(transc_path)
print(f"done  ling shape: {ling.shape}")

# ── Show transcript snippet ───────────────────────────────────────────────────
snippet = text[:220].strip()
display(HTML(
    "<div style='background:#1E293B;border-left:4px solid #C4501A;border-radius:6px;"
    "padding:12px 16px;margin:10px 0;font-family:Consolas;color:#CBD5E1;font-size:0.9em'>"
    "<div style='color:#F59E0B;font-weight:700;margin-bottom:6px'>Transcript snippet:</div>"
    "<em style='color:#94A3B8'>\"" + snippet + "...\"</em></div>"
))

# ── Apply fitted preprocessors ────────────────────────────────────────────────
Xsm = sm_pca.transform(sm_sc.transform(sm.reshape(1,-1)))
Xlb = lb_pca.transform(lb_sc.transform(lb.reshape(1,-1)))
Xtf = tf_pca.transform(tf_sc.transform(vec.transform([text]).toarray()))
Xlg = lg_sc.transform(ling.reshape(1,-1))
X_audio = np.hstack([Xsm, Xlb, Xtf, Xlg])

print(f"\n✓ Preprocessed feature vector:")
print(f"  openSMILE (PCA)   : {Xsm.shape[1]} components")
print(f"  librosa (PCA)     : {Xlb.shape[1]} components")
print(f"  TF-IDF (PCA)      : {Xtf.shape[1]} components")
print(f"  Linguistic        : {Xlg.shape[1]} features")
print(f"  Fused input to XGBoost: {X_audio.shape[1]}-dim")

# ── XGBoost prediction ────────────────────────────────────────────────────────
print("\nRunning XGBoost audio classifier ...")
p_audio_demo = float(audio_model.predict_proba(X_audio)[0, 1])

print()
show_result("P(dementia | Speech)", p_audio_demo, demo_true_audio)

# Save for Stage C
demo_bridge_audio = np.array([[demo_age_audio, 0.0, demo_mmse_audio]])


STAGE B — AUDIO PIPELINE
✓ Located  : cc/S007.wav


Subject ID,S007
Folder,cc/ (cc=Healthy · cd=Dementia)
Age,71 years
MMSE,28
True label,HEALTHY



Extracting openSMILE ComParE (6,373 acoustic functionals) ... done  shape: (6373,)
Extracting librosa spectral features ... done  shape: (30,)
Parsing CHAT transcript + handcrafted linguistic features ... done  ling shape: (11,)



✓ Preprocessed feature vector:
  openSMILE (PCA)   : 65 components
  librosa (PCA)     : 16 components
  TF-IDF (PCA)      : 72 components
  Linguistic        : 11 features
  Fused input to XGBoost: 164-dim

Running XGBoost audio classifier ...



In [34]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import numpy as np
_aud_oof    = _meta["audio_oof_prob"]
_aud_labels = _meta["audio_labels"]
_mri_oof    = _meta["mri_oof_prob"]
_y_fusion   = _meta["y_fusion"]
_br_oasis   = _meta["br_oasis"]
_br_adress  = _meta["br_adress"]


print("=" * 60)
print(f"STAGE C — KNN CLINICAL BRIDGE  (K = {K})")
print("=" * 60)

# ── Helper: KNN lookup ────────────────────────────────────────────────────────
def run_knn(query_raw_age, query_raw_mmse,
            library_probs, library_labels,
            lib_raw_age, lib_raw_mmse, k):
    # Drop any NaN rows from library
    valid = np.isfinite(lib_raw_age) & np.isfinite(lib_raw_mmse)
    lib_age_v   = lib_raw_age[valid]
    lib_mmse_v  = lib_raw_mmse[valid]
    lib_probs_v = library_probs[valid]
    lib_labels_v = library_labels[valid]
    valid_idx   = np.where(valid)[0]   # original indices

    age_mean  = lib_age_v.mean();   age_std  = lib_age_v.std()  + 1e-6
    mmse_mean = lib_mmse_v.mean();  mmse_std = lib_mmse_v.std() + 1e-6

    lib_scaled = np.column_stack([
        (lib_age_v  - age_mean)  / age_std,
        (lib_mmse_v - mmse_mean) / mmse_std,
    ])
    qry_scaled = np.array([[
        (query_raw_age  - age_mean)  / age_std,
        (query_raw_mmse - mmse_mean) / mmse_std,
    ]])

    nn = NearestNeighbors(n_neighbors=k, metric="euclidean").fit(lib_scaled)
    dists, idxs = nn.kneighbors(qry_scaled)
    d   = dists[0];  idx_local = idxs[0]
    idx = valid_idx[idx_local]   # map back to original indices
    w   = 1.0 / (d + 1e-6);  w /= w.sum()
    p_knn     = float(np.dot(w, lib_probs_v[idx_local]))
    p_dem     = float(lib_labels_v[idx_local].mean())
    entropy   = float(-p_dem * np.log(p_dem + 1e-9)
                      - (1-p_dem) * np.log(1-p_dem + 1e-9))
    mean_dist = float(d.mean())
    return p_knn, entropy, mean_dist, idx, d, w

# ── Helper: entropy colour and label ─────────────────────────────────────────
def ent_fmt(entropy):
    col   = ("#EF4444" if entropy > 0.462
             else "#F59E0B" if entropy > 0.231
             else "#10B981")
    label = ("HIGH — MMSE overlap zone, imputation less trustworthy" if entropy > 0.462
             else "MODERATE — some label mixing in neighbourhood"     if entropy > 0.231
             else "LOW — neighbours agree, imputation reliable")
    return col, label

# ── Helper: summary footer row ────────────────────────────────────────────────
def summary_row(p_knn, entropy, mean_dist):
    ec, el = ent_fmt(entropy)
    return (
        f"<tr style='background:#0F172A'>"
        f"<td colspan='4' style='padding:8px 12px;color:#94A3B8;font-size:0.85em'>"
        f"P_knn = <span style='color:#8B5CF6;font-weight:700;"
        f"font-family:Consolas'>{p_knn:.4f}</span>"
        f"&nbsp;&nbsp;|&nbsp;&nbsp;entropy = "
        f"<span style='color:{ec};font-weight:700'>{entropy:.4f}</span>"
        f" ({el})"
        f"&nbsp;&nbsp;|&nbsp;&nbsp;mean dist = "
        f"<span style='color:#64748B'>{mean_dist:.4f}</span></td>"
        f"<td colspan='3'></td></tr>"
    )

# ── Helper: shared table header ───────────────────────────────────────────────
def table_header(title, accent, p_col_label):
    return (
        f"<div style='background:{accent}22;border-left:4px solid {accent};"
        f"padding:8px 14px;border-radius:4px 4px 0 0;color:{accent};"
        f"font-weight:700;font-family:Segoe UI,sans-serif'>{title}</div>"
        f"<table style='border-collapse:collapse;width:100%;"
        f"font-family:Segoe UI,sans-serif;font-size:0.9em'>"
        f"<tr style='background:#1E3A5F;color:#64748B;font-size:0.82em'>"
        f"<th style='padding:6px 12px'>Rank</th>"
        f"<th style='padding:6px 12px'>Age</th>"
        f"<th style='padding:6px 12px'>MMSE</th>"
        f"<th style='padding:6px 12px'>Distance</th>"
        f"<th style='padding:6px 12px'>Weight w\u2096</th>"
        f"<th style='padding:6px 12px'>{p_col_label}</th>"
        f"<th style='padding:6px 12px'>Label</th></tr>"
    )

def table_row(rank, age_n, mmse_n, di, wi, p_n, lbl_n):
    col = "#EF4444" if lbl_n else "#10B981"
    bg  = "#1A1F2E" if rank % 2 else "#1E293B"
    return (
        f"<tr style='background:{bg}'>"
        f"<td style='padding:5px 12px;color:#94A3B8;text-align:center'>{rank}</td>"
        f"<td style='padding:5px 12px;color:#CBD5E1;text-align:center'>{age_n:.0f}</td>"
        f"<td style='padding:5px 12px;color:#CBD5E1;text-align:center'>{mmse_n:.0f}</td>"
        f"<td style='padding:5px 12px;color:#64748B;text-align:center'>{di:.4f}</td>"
        f"<td style='padding:5px 12px;color:#38BDF8;text-align:center;"
        f"font-family:Consolas'>{wi:.4f}</td>"
        f"<td style='padding:5px 12px;color:{col};text-align:center;"
        f"font-family:Consolas;font-weight:700'>{p_n:.4f}</td>"
        f"<td style='padding:5px 12px;color:{col};text-align:center;"
        f"font-weight:700'>{'DEMENTIA' if lbl_n else 'HEALTHY'}</td>"
        f"</tr>"
    )

# ── CASE 1: MRI subject → find K nearest ADReSS neighbours ───────────────────
_meta_aud  = load_adress_meta(CC_META_PATH, CD_META_PATH).reset_index(drop=True)
_aud_ages  = _meta_aud["Age"].values.astype(float)
_aud_mmses = _meta_aud["MMSE"].values.astype(float)

print(f"\nCase 1: {DEMO_MRI_SUBJECT}  (MRI subject, no speech)")
print(f"  Query: Age={demo_age_mri:.0f}  MMSE={demo_mmse_mri:.0f}")
print(f"  Searching {len(_aud_ages)} ADReSS subjects ...\n")

(p_knn_demo, entropy_demo, dist_demo,
 idx_mri, d_mri, w_mri) = run_knn(
    demo_age_mri, demo_mmse_mri,
    _aud_oof, _aud_labels,
    _aud_ages, _aud_mmses, K)

rows_c1 = ""
for rank, (ni, di, wi) in enumerate(zip(idx_mri, d_mri, w_mri), 1):
    age_n  = float(_meta_aud.iloc[ni]["Age"])  if ni < len(_meta_aud) else 0
    mmse_n = float(_meta_aud.iloc[ni]["MMSE"]) if ni < len(_meta_aud) else 0
    rows_c1 += table_row(rank, age_n, mmse_n, di, wi,
                         _aud_oof[ni], int(_aud_labels[ni]))

display(HTML(
    f"<div style='margin:16px 0;background:#0F172A;"
    f"border:1px solid #334155;border-radius:8px'>"
    + table_header(
        f"Case 1 — K={K} Nearest ADReSS Neighbours "
        f"for MRI subject: {DEMO_MRI_SUBJECT}",
        "#38BDF8", "P_audio (OOF)")
    + rows_c1
    + summary_row(p_knn_demo, entropy_demo, dist_demo)
    + "</table></div>"
))

# ── CASE 2: Audio subject → find K nearest OASIS neighbours ──────────────────
# Drop NaN from OASIS Age/MMSE before building arrays
_oasis_clin  = pd.read_excel(OASIS_CLINICAL)
_oasis_clin["Gender"] = _oasis_clin["M/F"].map({"M": 0, "F": 1})

# Align with df_v (the subjects that have both FS + CNN — same as training)
_df_v_clean  = df_v[["ID","Age","MMSE"]].copy()
_df_v_clean["Age"]  = pd.to_numeric(_df_v_clean["Age"],  errors="coerce")
_df_v_clean["MMSE"] = pd.to_numeric(_df_v_clean["MMSE"], errors="coerce")
_df_v_clean = _df_v_clean.reset_index(drop=True)

_oasis_ages  = _df_v_clean["Age"].values.astype(float)
_oasis_mmses = _df_v_clean["MMSE"].values.astype(float)

print(f"Case 2: {DEMO_AUDIO_SUBJECT}  (Audio subject, no MRI)")
print(f"  Query: Age={demo_age_audio:.0f}  MMSE={demo_mmse_audio:.0f}")
print(f"  Searching {len(_oasis_ages)} OASIS subjects ...\n")

(p_knn_audio, entropy_audio, dist_audio,
 idx_aud, d_aud, w_aud) = run_knn(
    demo_age_audio, demo_mmse_audio,
    _mri_oof, _y_fusion,
    _oasis_ages, _oasis_mmses, K)

rows_c2 = ""
for rank, (ni, di, wi) in enumerate(zip(idx_aud, d_aud, w_aud), 1):
    age_n  = float(_df_v_clean.iloc[ni]["Age"])  if ni < len(_df_v_clean) else 0
    mmse_n = float(_df_v_clean.iloc[ni]["MMSE"]) if ni < len(_df_v_clean) else 0
    rows_c2 += table_row(rank, age_n, mmse_n, di, wi,
                         _mri_oof[ni], int(_y_fusion[ni]))

display(HTML(
    f"<div style='margin:16px 0;background:#0F172A;"
    f"border:1px solid #334155;border-radius:8px'>"
    + table_header(
        f"Case 2 — K={K} Nearest OASIS Neighbours "
        f"for Audio subject: {DEMO_AUDIO_SUBJECT}",
        "#FB923C", "P_mri (OOF)")
    + rows_c2
    + summary_row(p_knn_audio, entropy_audio, dist_audio)
    + "</table></div>"
))

print("Both KNN lookups complete. Run Demo Cell 5 for final predictions.")

STAGE C — KNN CLINICAL BRIDGE  (K = 7)

Case 1: OAS1_0425_MR1  (MRI subject, no speech)
  Query: Age=78  MMSE=23
  Searching 108 ADReSS subjects ...



Case 2: S007  (Audio subject, no MRI)
  Query: Age=71  MMSE=28
  Searching 425 OASIS subjects ...



Both KNN lookups complete. Run Demo Cell 5 for final predictions.


In [35]:
print("=" * 60)
print("BRIDGE VECTOR  ->  KNN META-LEARNER  ->  FINAL PREDICTIONS")
print("=" * 60)

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neighbors import NearestNeighbors

# ── Rebuild KNN meta-learner from OOF data ────────────────────────────────────
_N_mri = len(_mri_oof); _N_aud = len(_aud_oof)

def _knn_imp_full(qry_br, lib_br, lib_probs, lib_labels, k):
    sc  = StandardScaler()
    lib = sc.fit_transform(lib_br[:, [0, 2]])
    qry = sc.transform(qry_br[:, [0, 2]])
    nn  = NearestNeighbors(n_neighbors=k, metric="euclidean").fit(lib)
    dists, idxs = nn.kneighbors(qry)
    imp = np.zeros(len(qry)); ent = np.zeros(len(qry)); md = np.zeros(len(qry))
    for i in range(len(qry)):
        d = dists[i]; idx = idxs[i]
        w = 1.0 / (d + 1e-6); w /= w.sum()
        imp[i] = np.dot(w, lib_probs[idx])
        p = lib_labels[idx].mean()
        ent[i] = -p * np.log(p + 1e-9) - (1-p) * np.log(1-p + 1e-9)
        md[i]  = d.mean()
    return imp, ent, md

_p_aud_knn, _ent_o, _dist_o = _knn_imp_full(
    _br_oasis, _br_adress, _aud_oof, _aud_labels, K)
_p_mri_knn, _ent_a, _dist_a = _knn_imp_full(
    _br_adress, _br_oasis, _mri_oof, _y_fusion, K)

X_knn = np.vstack([
    np.column_stack([_mri_oof,         _p_aud_knn,
                     np.ones(_N_mri),  np.zeros(_N_mri), _ent_o, _dist_o]),
    np.column_stack([_p_mri_knn,       _aud_oof,
                     np.zeros(_N_aud), np.ones(_N_aud),  _ent_a, _dist_a]),
])
y_knn = np.concatenate([_y_fusion, _aud_labels])
knn_sc   = StandardScaler()
X_knn_sc = knn_sc.fit_transform(X_knn)
knn_meta = CalibratedClassifierCV(
    LogisticRegression(max_iter=2000, class_weight="balanced",
                       C=0.5, random_state=42),
    cv=5, method="isotonic"
)
knn_meta.fit(X_knn_sc, y_knn)
print("KNN meta-learner ready.\n")

# ── Case 1 bridge vector ──────────────────────────────────────────────────────
bridge_mri = np.array([[
    p_mri_demo,    # P_real    — MRI pipeline (measured)
    p_knn_demo,    # P_knn     — imputed audio from ADReSS neighbours
    1.0,           # has_mri
    0.0,           # has_audio
    entropy_demo,  # knn_entropy
    dist_demo,     # knn_dist
]])

# ── Case 2 bridge vector ──────────────────────────────────────────────────────
bridge_audio = np.array([[
    p_audio_demo,   # P_real    — audio pipeline (measured)
    p_knn_audio,    # P_knn     — imputed MRI from OASIS neighbours
    0.0,            # has_mri
    1.0,            # has_audio
    entropy_audio,  # knn_entropy
    dist_audio,     # knn_dist
]])

# ── Predict both ──────────────────────────────────────────────────────────────
p_final_mri   = float(knn_meta.predict_proba(knn_sc.transform(bridge_mri))[0, 1])
p_final_audio = float(knn_meta.predict_proba(knn_sc.transform(bridge_audio))[0, 1])

# ── Bridge vector comparison table ───────────────────────────────────────────
feat_rows = [
    ("P_real",
     f"{p_mri_demo:.4f}",
     f"{p_audio_demo:.4f}",
     "#2E75B6",
     "Own modality probability (measured)"),
    ("P_knn",
     f"{p_knn_demo:.4f}",
     f"{p_knn_audio:.4f}",
     "#8B5CF6",
     "Imputed missing modality via KNN weighted average"),
    ("has_mri",
     "1.0",
     "0.0",
     "#1A7B8A",
     "Binary flag — does subject have MRI?"),
    ("has_audio",
     "0.0",
     "1.0",
     "#1A7B8A",
     "Binary flag — does subject have speech?"),
    ("knn_entropy",
     f"{entropy_demo:.4f}",
     f"{entropy_audio:.4f}",
     "#EF4444",
     "Label mix of K neighbours (0=pure, 0.693=max uncertainty)"),
    ("knn_dist",
     f"{dist_demo:.4f}",
     f"{dist_audio:.4f}",
     "#F59E0B",
     "Mean Euclidean distance to K neighbours"),
]

feat_html = ""
for name, v1, v2, col, desc in feat_rows:
    feat_html += (
        f"<tr style='border-bottom:1px solid #1E293B'>"
        f"<td style='padding:7px 14px;color:{col};font-family:Consolas;"
        f"font-weight:700;white-space:nowrap'>{name}</td>"
        f"<td style='padding:7px 14px;color:#38BDF8;font-family:Consolas;"
        f"text-align:center'>{v1}</td>"
        f"<td style='padding:7px 14px;color:#FB923C;font-family:Consolas;"
        f"text-align:center'>{v2}</td>"
        f"<td style='padding:7px 14px;color:#64748B;font-size:0.85em'>{desc}</td>"
        f"</tr>"
    )

display(HTML(
    "<div style='background:#0F172A;border:1px solid #334155;"
    "border-radius:8px;margin:14px 0;font-family:Segoe UI,sans-serif'>"
    "<div style='padding:10px 14px;color:#1A7B8A;font-weight:700;"
    "border-bottom:1px solid #334155'>"
    "6-Feature Bridge Vector -- Age / Gender / MMSE excluded (zero redundancy)</div>"
    "<table style='border-collapse:collapse;width:100%'>"
    "<tr style='background:#1E3A5F;color:#64748B;font-size:0.82em'>"
    "<th style='padding:6px 14px;text-align:left'>Feature</th>"
    "<th style='padding:6px 14px;text-align:center;color:#38BDF8'>"
    "Case 1: MRI subject<br><small style='font-weight:400'>"
    + DEMO_MRI_SUBJECT +
    "<br>P_real=MRI, P_knn=imputed audio</small></th>"
    "<th style='padding:6px 14px;text-align:center;color:#FB923C'>"
    "Case 2: Audio subject<br><small style='font-weight:400'>"
    + DEMO_AUDIO_SUBJECT +
    "<br>P_real=audio, P_knn=imputed MRI</small></th>"
    "<th style='padding:6px 14px;text-align:left'>Description</th>"
    "</tr>"
    + feat_html +
    "</table></div>"
))

# ── Final results table ───────────────────────────────────────────────────────
def result_row(case_label, subject, p_final, true_label, accent):
    verdict = "DEMENTIA" if p_final > 0.5 else "HEALTHY"
    correct = (p_final > 0.5) == bool(true_label)
    pcol    = "#EF4444" if p_final > 0.5 else "#10B981"
    tcol    = "#10B981" if correct else "#EF4444"
    conf    = ("HIGH"     if abs(p_final - 0.5) > 0.3
               else "MODERATE" if abs(p_final - 0.5) > 0.15
               else "LOW")
    return (
        f"<tr style='border-bottom:1px solid #1E293B'>"
        f"<td style='padding:10px 14px;color:{accent};font-weight:700'>"
        f"{case_label}<br>"
        f"<small style='color:#64748B;font-weight:400'>{subject}</small></td>"
        f"<td style='padding:10px 14px;color:#38BDF8;font-family:Consolas;"
        f"font-weight:700;text-align:center;font-size:1.2em'>{p_final:.4f}</td>"
        f"<td style='padding:10px 14px;color:{pcol};font-weight:700;"
        f"text-align:center'>{verdict}</td>"
        f"<td style='padding:10px 14px;color:#64748B;text-align:center'>"
        f"{'DEMENTIA' if true_label else 'HEALTHY'}</td>"
        f"<td style='padding:10px 14px;color:{tcol};font-weight:700;"
        f"text-align:center'>{'correct' if correct else 'wrong'}</td>"
        f"<td style='padding:10px 14px;color:#64748B;text-align:center'>"
        f"{conf}</td>"
        f"</tr>"
    )

results_html = (
    result_row(
        "Case 1 — MRI subject",
        f"{DEMO_MRI_SUBJECT} | P_real=MRI | P_knn=imputed audio",
        p_final_mri, demo_true_mri, "#38BDF8"
    ) +
    result_row(
        "Case 2 — Audio subject",
        f"{DEMO_AUDIO_SUBJECT} | P_real=audio | P_knn=imputed MRI",
        p_final_audio, demo_true_audio, "#FB923C"
    )
)

display(HTML(
    "<div style='background:#0F172A;border:1px solid #334155;"
    "border-radius:8px;margin:16px 0;font-family:Segoe UI,sans-serif'>"
    "<div style='padding:10px 14px;color:#8B5CF6;font-weight:700;"
    "border-bottom:1px solid #334155'>"
    "Final Fused Predictions -- KNN Clinical Bridge Meta-Learner</div>"
    "<table style='border-collapse:collapse;width:100%'>"
    "<tr style='background:#1E3A5F;color:#64748B;font-size:0.82em'>"
    "<th style='padding:6px 14px;text-align:left'>Subject</th>"
    "<th style='padding:6px 14px;text-align:center'>P(dementia)</th>"
    "<th style='padding:6px 14px;text-align:center'>Prediction</th>"
    "<th style='padding:6px 14px;text-align:center'>Ground Truth</th>"
    "<th style='padding:6px 14px;text-align:center'>Result</th>"
    "<th style='padding:6px 14px;text-align:center'>Confidence</th>"
    "</tr>"
    + results_html +
    "</table></div>"
))



BRIDGE VECTOR  ->  KNN META-LEARNER  ->  FINAL PREDICTIONS
KNN meta-learner ready.



Feature,"Case 1: MRI subjectOAS1_0425_MR1P_real=MRI, P_knn=imputed audio","Case 2: Audio subjectS007P_real=audio, P_knn=imputed MRI",Description
P_real,0.9959,0.0915,Own modality probability (measured)
P_knn,0.7771,0.5338,Imputed missing modality via KNN weighted average
has_mri,1.0,0.0,Binary flag — does subject have MRI?
has_audio,0.0,1.0,Binary flag — does subject have speech?
knn_entropy,0.6829,0.6829,"Label mix of K neighbours (0=pure, 0.693=max uncertainty)"
knn_dist,0.6017,0.1690,Mean Euclidean distance to K neighbours


Subject,P(dementia),Prediction,Ground Truth,Result,Confidence
Case 1 — MRI subjectOAS1_0425_MR1 | P_real=MRI | P_knn=imputed audio,0.8224,DEMENTIA,DEMENTIA,correct,HIGH
Case 2 — Audio subjectS007 | P_real=audio | P_knn=imputed MRI,0.4991,HEALTHY,HEALTHY,correct,LOW


In [136]:
print("=" * 60)
print("COMPLETE PIPELINE SUMMARY")
print("=" * 60)

# ── Per-stage rows ────────────────────────────────────────────────────────────
summary_rows = [
    (
        "Stage A — MRI Pipeline",
        DEMO_MRI_SUBJECT,
        "FreeSurfer + 3D CNN + Clinical → XGBoost",
        p_mri_demo,
        demo_true_mri,
        "#38BDF8"
    ),
    (
        "Stage B — Audio Pipeline",
        DEMO_AUDIO_SUBJECT,
        "openSMILE + librosa + TF-IDF + Linguistic → XGBoost",
        p_audio_demo,
        demo_true_audio,
        "#FB923C"
    ),
    (
        "Stage C — (MRI query)",
        DEMO_MRI_SUBJECT,
        f"P_real=MRI · P_knn=imputed audio · K={K} ADReSS neighbours",
        p_final_mri,
        demo_true_mri,
        "#8B5CF6"
    ),
    (
        "Stage C — (Audio query)",
        DEMO_AUDIO_SUBJECT,
        f"P_real=audio · P_knn=imputed MRI · K={K} OASIS neighbours",
        p_final_audio,
        demo_true_audio,
        "#A78BFA"
    ),
]

rows_html = ""
for stage, subject, method, prob, true_label, accent in summary_rows:
    verdict = "DEMENTIA" if prob > 0.5 else "HEALTHY"
    correct = (prob > 0.5) == bool(true_label)
    pcol    = "#EF4444" if prob > 0.5 else "#10B981"
    tcol    = "#10B981" if correct else "#EF4444"
    conf    = ("HIGH"     if abs(prob - 0.5) > 0.3
               else "MODERATE" if abs(prob - 0.5) > 0.15
               else "LOW")
    rows_html += (
        f"<tr style='border-bottom:1px solid #1E293B'>"
        f"<td style='padding:10px 14px;color:{accent};font-weight:700;"
        f"white-space:nowrap'>{stage}</td>"
        f"<td style='padding:10px 14px;color:#94A3B8;font-family:Consolas;"
        f"font-size:0.85em'>{subject}</td>"
        f"<td style='padding:10px 14px;color:#475569;font-size:0.82em'>{method}</td>"
        f"<td style='padding:10px 14px;color:#38BDF8;font-family:Consolas;"
        f"font-weight:700;text-align:center;font-size:1.1em'>{prob:.4f}</td>"
        f"<td style='padding:10px 14px;color:{pcol};font-weight:700;"
        f"text-align:center'>{verdict}</td>"
        f"<td style='padding:10px 14px;color:#64748B;text-align:center'>"
        f"{'DEMENTIA' if true_label else 'HEALTHY'}</td>"
        f"<td style='padding:10px 14px;color:{tcol};font-weight:700;"
        f"text-align:center'>{'correct' if correct else 'wrong'}</td>"
        f"<td style='padding:10px 14px;color:#64748B;text-align:center'>{conf}</td>"
        f"</tr>"
    )

display(HTML(
    "<div style='background:#0F172A;border:1px solid #334155;"
    "border-radius:8px;margin:14px 0;font-family:Segoe UI,sans-serif'>"
    "<div style='padding:10px 14px;color:#1A7B8A;font-weight:700;"
    "border-bottom:1px solid #334155'>Full Pipeline Summary — All Stages</div>"
    "<table style='border-collapse:collapse;width:100%'>"
    "<tr style='background:#1E3A5F;color:#64748B;font-size:0.82em'>"
    "<th style='padding:6px 14px;text-align:left'>Stage</th>"
    "<th style='padding:6px 14px;text-align:left'>Subject</th>"
    "<th style='padding:6px 14px;text-align:left'>Method</th>"
    "<th style='padding:6px 14px;text-align:center'>P(dementia)</th>"
    "<th style='padding:6px 14px;text-align:center'>Prediction</th>"
    "<th style='padding:6px 14px;text-align:center'>Ground Truth</th>"
    "<th style='padding:6px 14px;text-align:center'>Result</th>"
    "<th style='padding:6px 14px;text-align:center'>Confidence</th>"
    "</tr>"
    + rows_html +
    "</table></div>"
))

# ── Overall OOF benchmark ─────────────────────────────────────────────────────
display(HTML(
    "<div style='background:#1E293B;border-radius:8px;padding:18px 22px;"
    "margin:16px 0;font-family:Segoe UI,sans-serif'>"
    "<div style='color:#94A3B8;font-size:0.82em;margin-bottom:14px;letter-spacing:1px'>"
    "OVERALL SYSTEM PERFORMANCE  (5-Fold OOF on all subjects)</div>"
    "<div style='display:flex;gap:32px;flex-wrap:wrap;align-items:flex-end'>"

    "<div style='border-left:3px solid #38BDF8;padding-left:14px'>"
    "<div style='color:#38BDF8;font-size:0.8em;margin-bottom:4px'>"
    "MRI Pipeline · N=425</div>"
    "<div style='color:#F1F5F9;font-size:1.9em;font-weight:700;"
    "font-family:Consolas'>0.9710</div>"
    "<div style='color:#64748B;font-size:0.8em'>AUC &nbsp;|&nbsp; ACC 88.94%</div>"
    "<div style='color:#475569;font-size:0.75em;margin-top:4px'>"
    "Exceeds published baseline 0.920</div></div>"

    "<div style='border-left:3px solid #FB923C;padding-left:14px'>"
    "<div style='color:#FB923C;font-size:0.8em;margin-bottom:4px'>"
    "Audio Pipeline · N=108</div>"
    "<div style='color:#F1F5F9;font-size:1.9em;font-weight:700;"
    "font-family:Consolas'>0.8920</div>"
    "<div style='color:#64748B;font-size:0.8em'>AUC &nbsp;|&nbsp; ACC 84.26%</div>"
    "<div style='color:#475569;font-size:0.75em;margin-top:4px'>"
    "Exceeds DementiaBank avg 0.813</div></div>"

    "<div style='border-left:3px solid #8B5CF6;padding-left:14px'>"
    "<div style='color:#8B5CF6;font-size:0.8em;margin-bottom:4px'>"
    "KNN Meta-Learner · N=533</div>"
    "<div style='color:#F1F5F9;font-size:1.9em;font-weight:700;"
    "font-family:Consolas'>0.9425</div>"
    "<div style='color:#64748B;font-size:0.8em'>AUC &nbsp;|&nbsp; ACC 90.24%</div>"
    "<div style='color:#475569;font-size:0.75em;margin-top:4px'>"
    "Combined 533 subjects, both datasets</div></div>"

    "</div></div>"
))

COMPLETE PIPELINE SUMMARY


Stage,Subject,Method,P(dementia),Prediction,Ground Truth,Result,Confidence
Stage A — MRI Pipeline,OAS1_0425_MR1,FreeSurfer + 3D CNN + Clinical → XGBoost,0.9959,DEMENTIA,DEMENTIA,correct,HIGH
Stage B — Audio Pipeline,S007,openSMILE + librosa + TF-IDF + Linguistic → XGBoost,0.0915,HEALTHY,HEALTHY,correct,HIGH
Stage C — (MRI query),OAS1_0425_MR1,P_real=MRI · P_knn=imputed audio · K=7 ADReSS neighbours,0.8224,DEMENTIA,DEMENTIA,correct,HIGH
Stage C — (Audio query),S007,P_real=audio · P_knn=imputed MRI · K=7 OASIS neighbours,0.4991,HEALTHY,HEALTHY,correct,LOW
